<font color="green">

# Домашнє завдання: Sentiment Analysis

</font>

### Мета:
1. Завантажити та підготувати набір даних `rt-polaritydata`.
2. Реалізувати щонайменше 3 методи класифікації тексту (пріоритет на сучасні підходи).
3. Порівняти метрики: Accuracy, Precision, Recall, F1-score.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

def load_data(filepath, label):
    with open(filepath, "r", encoding='utf-8', errors='ignore') as f:
        lines = f.read().splitlines()
    return pd.DataFrame({'text': lines, 'label': label})

df_neg = load_data('rt-polarity.neg', 0)
df_pos = load_data('rt-polarity.pos', 1)

print(f"Кількість негативних відгуків: {len(df_neg)}")
print(f"Кількість позитивних відгуків: {len(df_pos)}")

df = pd.concat([df_neg, df_pos], ignore_index=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

print(f"Розмір тренувальної вибірки: {len(X_train)}")
print(f"Розмір тестової вибірки: {len(X_test)}")

def print_metrics(y_true, y_pred, model_name):
    print(f"--- {model_name} ---")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1-score:  {f1_score(y_true, y_pred):.4f}\n")

Кількість негативних відгуків: 5331
Кількість позитивних відгуків: 5331
Розмір тренувальної вибірки: 8529
Розмір тестової вибірки: 2133


<font color="green">

### Метод 1: N-grams + Logistic Regression

</font>
Використаємо `CountVectorizer` з біграмами (n-gram range 1, 2) та логістичну регресію як сильний класичний бейзлайн.

In [2]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

ngram_vect = CountVectorizer(min_df=3, max_features=20000, ngram_range=(1,2))
X_train_ng = ngram_vect.fit_transform(X_train)
X_test_ng = ngram_vect.transform(X_test)

clf_ng = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_ng, y_train)

preds_ng = clf_ng.predict(X_test_ng)
print_metrics(y_test, preds_ng, "N-grams (1,2) + Logistic Regression")

--- N-grams (1,2) + Logistic Regression ---
Accuracy:  0.7707
Precision: 0.7782
Recall:    0.7570
F1-score:  0.7675



<font color="green">

### Метод 2: TF-IDF + Logistic Regression

</font>
Використаємо `TfidfVectorizer`, який зважує слова залежно від їхньої важливості у документі та корпусі.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer(min_df=3, max_features=20000)
X_train_tfidf = tfidf_vect.fit_transform(X_train)
X_test_tfidf = tfidf_vect.transform(X_test)

clf_tfidf = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_tfidf, y_train)

preds_tfidf = clf_tfidf.predict(X_test_tfidf)
print_metrics(y_test, preds_tfidf, "TF-IDF + Logistic Regression")

--- TF-IDF + Logistic Regression ---
Accuracy:  0.7529
Precision: 0.7594
Recall:    0.7402
F1-score:  0.7496



<font color="green">

### Метод 3: Word Embeddings (Word2Vec) + Logistic Regression

</font>
Сучасний підхід: навчаємо власні векторні представлення слів за допомогою Word2Vec, усереднюємо вектори слів для кожного відгуку і передаємо їх у класифікатор.

In [4]:
import ssl
import nltk

# Ігноруємо перевірку SSL-сертифіката для завантаження
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec

nltk.download('punkt', quiet=True)

X_train_tok = [word_tokenize(text.lower()) for text in X_train]
X_test_tok = [word_tokenize(text.lower()) for text in X_test]

w2v_model = Word2Vec(X_train_tok, vector_size=100, window=5, min_count=2, workers=4)

def document_vector(doc, model):
    words = [word for word in doc if word in model.wv]
    if len(words) == 0:
        return np.zeros(model.vector_size)
    return np.mean([model.wv[word] for word in words], axis=0)

X_train_w2v = np.array([document_vector(doc, w2v_model) for doc in X_train_tok])
X_test_w2v = np.array([document_vector(doc, w2v_model) for doc in X_test_tok])

clf_w2v = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_w2v, y_train)

preds_w2v = clf_w2v.predict(X_test_w2v)
print_metrics(y_test, preds_w2v, "Word2Vec + Logistic Regression")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


--- Word2Vec + Logistic Regression ---
Accuracy:  0.5659
Precision: 0.5600
Recall:    0.6126
F1-score:  0.5851



<font color="green">

### Метод 4: Pre-trained Transformer (Hugging Face Pipeline)

</font>
Використання попередньо навченої моделі-трансформера (Zero-shot / Pre-trained). Для швидкості використовуємо базову модель `distilbert-base-uncased-finetuned-sst-2-english`, яка чудово підходить для аналізу тональності. 
*(Примітка: обробка понад 2000 тестових записів може зайняти кілька хвилин залежно від наявності GPU).*

In [5]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

classifier = pipeline("sentiment-analysis", device=device)

print("Прогнозування за допомогою Transformer (може зайняти кілька хвилин)...")
hf_results = classifier(X_test.tolist(), truncation=True, max_length=512)

preds_transformer = [1 if res['label'] == 'POSITIVE' else 0 for res in hf_results]

print_metrics(y_test, preds_transformer, "Pre-trained Transformer")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7013.55it/s]


Прогнозування за допомогою Transformer (може зайняти кілька хвилин)...
--- Pre-trained Transformer ---
Accuracy:  0.8856
Precision: 0.8806
Recall:    0.8921
F1-score:  0.8863



<font color="green">

### Підсумкове порівняння результатів

</font>
Зберемо всі результати в єдину таблицю для зручного аналізу.

In [6]:
results_dict = {
    'Model': ['N-Grams + LogReg', 'TF-IDF + LogReg', 'Word2Vec + LogReg', 'Pre-trained Transformer'],
    'Accuracy': [
        accuracy_score(y_test, preds_ng),
        accuracy_score(y_test, preds_tfidf),
        accuracy_score(y_test, preds_w2v),
        accuracy_score(y_test, preds_transformer)
    ],
    'Precision': [
        precision_score(y_test, preds_ng),
        precision_score(y_test, preds_tfidf),
        precision_score(y_test, preds_w2v),
        precision_score(y_test, preds_transformer)
    ],
    'Recall': [
        recall_score(y_test, preds_ng),
        recall_score(y_test, preds_tfidf),
        recall_score(y_test, preds_w2v),
        recall_score(y_test, preds_transformer)
    ],
    'F1-score': [
        f1_score(y_test, preds_ng),
        f1_score(y_test, preds_tfidf),
        f1_score(y_test, preds_w2v),
        f1_score(y_test, preds_transformer)
    ]
}

results_df = pd.DataFrame(results_dict)
results_df.sort_values(by='F1-score', ascending=False, inplace=True)
results_df.reset_index(drop=True, inplace=True)

display(results_df)

,Model,Accuracy,Precision,Recall,F1-score
0,Pre-trained Transformer,0.885607,0.880556,0.892120,0.886300
1,N-Grams + LogReg,0.770745,0.778206,0.757036,0.767475
2,TF-IDF + LogReg,0.752930,0.759384,0.740150,0.749644
3,Word2Vec + LogReg,0.565870,0.560034,0.612570,0.585125
